# XB3_report (local CPU): B report, saved-layer cross-check, section-3 rule

In [ ]:
XS_DIR = ""


In [ ]:
import glob, hashlib, json, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
PT3 = MYDRIVE / "ez-penta" / "post_train_3"
XS = Path(XS_DIR) if XS_DIR else PT3 / "x_steer"


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def folder_ok(d):
    d = Path(d)
    if not (d / "DONE").exists() or not (d / "manifest.json").exists():
        return False
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


# reference categorical palette (dataviz skill), fixed per entity
COL = {"base": "#2a78d6", "en_s0": "#eb6834", "ja_s0": "#1baf7a", "bi_s0": "#eda100", "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df"}
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": COL["ink2"],
                     "axes.labelcolor": COL["ink"], "xtick.color": COL["ink2"], "ytick.color": COL["ink2"], "axes.grid": True,
                     "grid.color": COL["grid"], "grid.linewidth": 0.6, "lines.linewidth": 1.6, "font.size": 9, "legend.frameon": False})


In [ ]:
B = XS / "B"
BD = B / "b_all_layers"
assert folder_ok(BD), "B diagnostics not downloaded/complete"
assert folder_ok(B / "s1_labels_ja_s0")
J = json.loads((BD / "diagnostics.json").read_text())
DG = pd.DataFrame(J["diag"].values())
MODELS = list(dict.fromkeys(DG.model))
LAYERS = J["layers"]
NL = J["num_hidden_layers"]
CROSS = pd.DataFrame(J["cross_model"].values())
print(MODELS, NL, J["counts"], J["low_n"], J.get("norm_info"))
FIG = B / "figures"
FIG.mkdir(exist_ok=True)
MAN = json.loads((BD / "manifest.json").read_text())
HW = {"hardware": MAN["hardware"], "gpu_name": MAN["gpu_name"], "versions": MAN["versions"]}
print(HW)


In [ ]:
# Saved-layer cross-check against steering (layers 8..28, base and en_s0): steering-equivalent cos(f,h) and in-sample AUC of f-perp,
# plus the activations themselves when they were downloaded (B/acts).
ST = json.loads((PT3 / "steering" / "s2_directions" / "diagnostics.json").read_text())
SAVED = [8, 12, 16, 18, 20, 24, 28]
rows = []
ACTL = B / "acts"
for m in ("base", "en_s0"):
    for L in SAVED:
        d = J["diag"][f"{m}|L{L}"]
        s = ST[f"{m}|L{L}"]
        r = {"model": m, "layer": L, "cos_f_h_B": d["steer_cos_f_h"], "cos_f_h_steer": s["cos_f_h"],
             "auc_fperp_B": d["steer_auc_fperp"], "auc_fperp_steer": s["auc_fperp_refused_vs_answered"],
             "auc_h_B": d["auc_in_h_harmful_vs_twin"], "auc_h_steer": s["auc_h_harmful_vs_twin"]}
        r["d_cos"] = r["cos_f_h_B"] - r["cos_f_h_steer"]
        r["d_auc_fperp"] = r["auc_fperp_B"] - r["auc_fperp_steer"]
        r["d_auc_h"] = r["auc_h_B"] - r["auc_h_steer"]
        fa, fb = ACTL / f"acts_{m}_L{L}.npy", PT3 / "steering" / "s2_directions" / "acts" / f"acts_{m}_L{L}.npy"
        if fa.exists():
            a, b = np.load(fa).astype(np.float32), np.load(fb).astype(np.float32)
            rc = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
            r["act_row_cos_min"], r["act_row_cos_mean"] = float(rc.min()), float(rc.mean())
            r["act_rel_diff"] = float(np.linalg.norm(a - b) / np.linalg.norm(b))
        r["flag"] = bool(abs(r["d_cos"]) > 0.05 or abs(r["d_auc_fperp"]) > 0.03)
        rows.append(r)
XC = pd.DataFrame(rows)
XC.to_csv(B / "crosscheck_steering.csv", index=False)
print(XC.round(4).to_string())
print("max |d_cos| %.4f, max |d_auc_fperp| %.4f, flagged %d" % (XC.d_cos.abs().max(), XC.d_auc_fperp.abs().max(), XC.flag.sum()))


In [ ]:
# XSTEER_SHARED section 3 (fixed in advance): en_s0, English. qualifies = held-out within-English AUC of f_en_perp_hg >= 0.80 and
# |cos(f_en, h_en)| <= 0.30. L* = qualifying layer with the highest held-out AUC (ties -> lower layer); else argmax(AUC - |cos|) as a
# negative control. low_n cells are excluded from decisions.
E = DG[DG.model == "en_s0"].sort_values("layer")
assert not J["low_n"]["en_s0"]["en"], "en_s0 English is low_n"
QL = []
for _, r in E.iterrows():
    a, c = r["auc_ho_f_en_perp_hg_en"], r["cos_f_en_h_en"]
    ok = a is not None and c is not None and not np.isnan(a) and not np.isnan(c)
    QL.append({"layer": int(r["layer"]), "auc_heldout_f_en_perp_hg_en": None if not ok else float(a), "cos_f_en_h_en": None if not ok else float(c),
               "abs_cos": None if not ok else abs(float(c)), "auc_insample_f_en_perp_hg_en": r["auc_in_f_en_perp_hg_en"],
               "qualifies": bool(ok and a >= 0.80 and abs(c) <= 0.30), "score_auc_minus_abscos": None if not ok else float(a - abs(c))})
Q = [q for q in QL if q["qualifies"]]
if Q:
    best = max(Q, key=lambda q: (round(q["auc_heldout_f_en_perp_hg_en"], 12), -q["layer"]))
    neg = False
    why = "qualifying layer with the highest held-out AUC"
else:
    cand = [q for q in QL if q["score_auc_minus_abscos"] is not None]
    best = max(cand, key=lambda q: (round(q["score_auc_minus_abscos"], 12), -q["layer"]))
    neg = True
    why = "no layer qualifies: layer maximising held-out AUC - |cos(f_en, h_en)| (negative control)"
LSTAR = best["layer"]
qual = {"rule": "XSTEER_SHARED section 3", "thresholds": {"auc_heldout_min": 0.80, "abs_cos_max": 0.30}, "source": "B (all layers)",
        "L_star": LSTAR, "negative_control": neg, "why": why, "n_qualifying": len(Q), "qualifying_layers": [q["layer"] for q in Q],
        "L_star_values": best, "layers": QL}
(B / "qualifying_layers.json").write_text(json.dumps(qual, indent=1))
print(json.dumps({k: qual[k] for k in ("L_star", "negative_control", "why", "n_qualifying", "qualifying_layers", "L_star_values")}, indent=1))


In [ ]:
# Hypothesis-H checks per layer.
def v(m, L, k):
    x = J["diag"][f"{m}|L{L}"].get(k)
    return np.nan if x is None else x


TL = {"en_s0": "en", "ja_s0": "ja"}
HT = []
for L in LAYERS:
    r = {"layer": L}
    for m in MODELS:
        for lang in ("en", "ja"):
            r[f"auc_ho_f_{lang}[{m}]_{lang}"] = v(m, L, f"auc_ho_f_{lang}_{lang}")
            r[f"auc_ho_f_{lang}_perp_hg[{m}]_{lang}"] = v(m, L, f"auc_ho_f_{lang}_perp_hg_{lang}")
        r[f"auc_ho_g[{m}]_all"] = v(m, L, "auc_ho_g_all")
        r[f"cos_f_en_f_ja[{m}]"] = v(m, L, "cos_f_en_f_ja")
        r[f"cos_h_en_h_ja[{m}]"] = v(m, L, "cos_h_en_h_ja")
        r[f"cos_f_en_h_en[{m}]"] = v(m, L, "cos_f_en_h_en")
        r[f"cos_f_ja_h_ja[{m}]"] = v(m, L, "cos_f_ja_h_ja")
    # training-language comparison: ja_s0's f_ja within Japanese vs en_s0's f_en within English (held-out, raw and perp_hg)
    r["train_lang_gap_raw"] = r["auc_ho_f_ja[ja_s0]_ja"] - r["auc_ho_f_en[en_s0]_en"]
    r["train_lang_gap_perp_hg"] = r["auc_ho_f_ja_perp_hg[ja_s0]_ja"] - r["auc_ho_f_en_perp_hg[en_s0]_en"]
    HT.append(r)
HT = pd.DataFrame(HT).merge(CROSS, on="layer")
HT.to_csv(B / "hypothesis_H_by_layer.csv", index=False)
COUNTS = pd.DataFrame([{"model": m, "lang": l, **J["counts"][m][l], "low_n": J["low_n"][m][l]} for m in MODELS for l in ("en", "ja")])
print(COUNTS.to_string())
# each adapter's AUC in its training language and in the other language (held-out, f_lang_perp_hg and raw f_lang)
TRAIN = []
for m in MODELS:
    for lang in ("en", "ja"):
        s = DG[DG.model == m].set_index("layer")
        TRAIN.append({"model": m, "lang": lang, "low_n": J["low_n"][m][lang],
                      "max_auc_ho_f_raw": float(s[f"auc_ho_f_{lang}_{lang}"].max()), "argmax_layer_raw": int(s[f"auc_ho_f_{lang}_{lang}"].idxmax()),
                      "max_auc_ho_f_perp_hg": float(s[f"auc_ho_f_{lang}_perp_hg_{lang}"].max()), "argmax_layer_perp_hg": int(s[f"auc_ho_f_{lang}_perp_hg_{lang}"].idxmax()),
                      "median_auc_ho_f_perp_hg": float(s[f"auc_ho_f_{lang}_perp_hg_{lang}"].median()),
                      "min_abs_cos_f_h_lang": float(s[f"cos_f_{lang}_h_{lang}"].abs().min()),
                      "argmin_layer_abs_cos": int(s[f"cos_f_{lang}_h_{lang}"].abs().idxmin()),
                      "max_auc_ho_g_all": float(s["auc_ho_g_all"].max())})
TRAIN = pd.DataFrame(TRAIN)
print(TRAIN.round(3).to_string())
# Pareto view: per model x language, layers with held-out AUC(f_lang_perp_hg) >= 0.80 and |cos(f_lang, h_lang)| <= 0.30
PAR = {}
for m in MODELS:
    for lang in ("en", "ja"):
        s = DG[DG.model == m]
        ok = s[(s[f"auc_ho_f_{lang}_perp_hg_{lang}"] >= 0.80) & (s[f"cos_f_{lang}_h_{lang}"].abs() <= 0.30)]
        PAR[f"{m}|{lang}"] = [int(x) for x in ok.layer]
print("layers meeting the section-3 thresholds, per model x language:", PAR)


In [ ]:
# Figures (static PNG; categorical colour fixed per model).
def lines(ax, key, title, ylab, models=MODELS, hlines=()):
    for m in models:
        s = DG[DG.model == m].sort_values("layer")
        ax.plot(s.layer, s[key].astype(float), color=COL[m], label=m)
    for y, lab in hlines:
        ax.axhline(y, color=COL["ink2"], lw=1, ls="--")
        ax.annotate(lab, (0, y), xytext=(2, 3), textcoords="offset points", color=COL["ink2"], fontsize=7)
    ax.set_title(title, fontsize=9, loc="left", color=COL["ink"])
    ax.set_xlabel("decoder layer")
    ax.set_ylabel(ylab)


# 1. section-3 rule for en_s0
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
lines(ax[0], "auc_ho_f_en_perp_hg_en", "Held-out AUC of f_en⊥hg within English", "AUC (refused vs answered twins)", hlines=[(0.80, "threshold 0.80")])
DG["abs_cos_f_en_h_en"] = DG.cos_f_en_h_en.abs()
lines(ax[1], "abs_cos_f_en_h_en", "|cos(f_en, h_en)|", "|cosine|", hlines=[(0.30, "threshold 0.30")])
for a in ax:
    a.axvline(LSTAR, color=COL["ink"], lw=1, ls=":")
ax[0].legend(loc="lower right", fontsize=7)
fig.suptitle(f"Section-3 rule (decision uses en_s0 only): L* = {LSTAR}{' (negative control)' if qual['negative_control'] else ''}", fontsize=9, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(FIG / "rule_en_s0_by_layer.png")
plt.close(fig)

# 2. within-language AUC of f_lang_perp_hg, each model, both languages
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
lines(ax[0], "auc_ho_f_en_perp_hg_en", "English: held-out AUC of f_en⊥hg", "AUC")
lines(ax[1], "auc_ho_f_ja_perp_hg_ja", "Japanese: held-out AUC of f_ja⊥hg", "AUC")
ax[0].legend(loc="lower right", fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "within_language_auc.png")
plt.close(fig)

# 3. language-only baseline AUC (projection on g, pooled twins) and cos(f_en, f_ja)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
lines(ax[0], "auc_ho_g_all", "Language-only baseline: held-out AUC of g, pooled twins", "AUC", hlines=[(0.5, "chance")])
lines(ax[1], "cos_f_en_f_ja", "cos(f_en, f_ja) within each model", "cosine")
ax[1].legend(fontsize=7)
ax[0].legend(loc="upper right", fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "language_baseline_and_fen_fja.png")
plt.close(fig)

# 4. cross-model cosines and harm sharing
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
cc = [("cos_f_en[en_s0]_f_ja[ja_s0]", "f_en[en_s0] vs f_ja[ja_s0]", "#2a78d6"), ("cos_f_en[en_s0]_f_en[ja_s0]", "f_en[en_s0] vs f_en[ja_s0]", "#eb6834"),
      ("cos_f_ja[en_s0]_f_ja[ja_s0]", "f_ja[en_s0] vs f_ja[ja_s0]", "#1baf7a")]
for k, lab, c in cc:
    s = CROSS.sort_values("layer")
    ax[0].plot(s.layer, s[k].astype(float), color=c, label=lab)
ax[0].set_title("Cross-model cosines of over-refusal directions", fontsize=9, loc="left")
ax[0].set_xlabel("decoder layer"); ax[0].set_ylabel("cosine"); ax[0].legend(fontsize=7)
lines(ax[1], "cos_h_en_h_ja", "Harm sharing: cos(h_en, h_ja)", "cosine")
ax[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "cross_model_and_harm_sharing.png")
plt.close(fig)
print(sorted(p.name for p in FIG.glob("*.png")))


In [ ]:
# report.md, results.json, manifest.json, DONE for PT3/x_steer/B.
def fm(x, d=3):
    return "–" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{d}f}"


SHOW = sorted(set([0, 4, 8, 12, 16, 18, 20, 24, 28, 32, NL - 1, LSTAR]) & set(LAYERS))
tab = "| layer | AUC_ho f_en⊥hg (en) | AUC_in | cos(f_en,h_en) | cos(f_en,h) | cos(f_en,g) | AUC_ho g (pooled) | qualifies |\n|---|---|---|---|---|---|---|---|\n"
for L in SHOW:
    d = J["diag"][f"en_s0|L{L}"]
    q = next(x for x in QL if x["layer"] == L)
    tab += (f"| {L}{' **L\\***' if L == LSTAR else ''} | {fm(d['auc_ho_f_en_perp_hg_en'])} | {fm(d['auc_in_f_en_perp_hg_en'])} | {fm(d['cos_f_en_h_en'], 2)} | "
            f"{fm(d['cos_f_en_h'], 2)} | {fm(d['cos_f_en_g'], 2)} | {fm(d['auc_ho_g_all'])} | {'yes' if q['qualifies'] else 'no'} |\n")
cnt = "| model | en refused/answered | ja refused/answered | low_n |\n|---|---|---|---|\n" + "".join(
    f"| {m} | {J['counts'][m]['en']['refused']}/{J['counts'][m]['en']['answered']} | {J['counts'][m]['ja']['refused']}/{J['counts'][m]['ja']['answered']} | "
    f"{', '.join(l for l in ('en', 'ja') if J['low_n'][m][l]) or '–'} |\n" for m in MODELS)
tr = "| model | lang | low_n | max AUC_ho f_lang (layer) | max AUC_ho f_lang⊥hg (layer) | median AUC_ho f_lang⊥hg | min abs cos(f_lang,h_lang) (layer) | max AUC_ho g |\n|---|---|---|---|---|---|---|---|\n" + "".join(
    f"| {r.model} | {r.lang} | {'yes' if r.low_n else 'no'} | {r.max_auc_ho_f_raw:.3f} (L{r.argmax_layer_raw}) | {r.max_auc_ho_f_perp_hg:.3f} (L{r.argmax_layer_perp_hg}) | "
    f"{r.median_auc_ho_f_perp_hg:.3f} | {r.min_abs_cos_f_h_lang:.2f} (L{r.argmin_layer_abs_cos}) | {r.max_auc_ho_g_all:.3f} |\n" for r in TRAIN.itertuples())
xs = "| layer | " + " | ".join(k.replace("cos_", "") for k, _, _ in cc) + " | cos(h_en,h_ja) en_s0 | cos(h_en,h_ja) ja_s0 |\n|" + "---|" * 6 + "\n"
for L in SHOW:
    c = J["cross_model"][f"L{L}"]
    xs += f"| {L} | " + " | ".join(fm(c[k], 2) for k, _, _ in cc) + f" | {fm(c['cos_h_en_h_ja[en_s0]'], 2)} | {fm(c['cos_h_en_h_ja[ja_s0]'], 2)} |\n"
xcm = XC[["model", "layer", "cos_f_h_B", "cos_f_h_steer", "auc_fperp_B", "auc_fperp_steer"] + [c for c in ("act_row_cos_mean", "act_rel_diff") if c in XC]].round(4)
xct = "| " + " | ".join(xcm.columns) + " |\n|" + "---|" * len(xcm.columns) + "\n" + "".join("| " + " | ".join(str(x) for x in r) + " |\n" for r in xcm.values)
lab_man = json.loads((B / "s1_labels_ja_s0" / "manifest.json").read_text())
rep = f"""# x_steer B: every layer, four models (Agent 2)

Hardware: {HW['hardware']} ({HW['gpu_name']}); versions {HW['versions']}. Models: {', '.join(MODELS)} (merged bf16). Layers 0–{NL - 1} (all decoder layers;
`hidden_states[L+1]` = output of layer L; norm check {J.get('norm_info')}). Extraction set: 3,864 prompts; {J['n_harm_pairs']} exact harmful/twin pairs,
{J['n_lang_pairs']} exact en/ja twin pairs. Held-out AUC = two seed folds by sha256(seed_id) % 2, fit on one, score the other, averaged.

## Section-3 rule (en_s0, English)

**L\\* = {LSTAR}; {qual['why']}.** Qualifying layers: {qual['qualifying_layers'] or 'none'}.

{tab}
## Refused-twin counts (0.6B judge)

{cnt}
ja_s0 labels (this session, XB1): {lab_man['refused_twins_model_lang']}, unparsed {lab_man['n_unparsed_g06']}, truncated {lab_man.get('n_truncated')}.

## Each adapter's over-refusal readability by language (held-out)

{tr}
Layers meeting both section-3 thresholds, per model × language: {PAR}

## Cross-model cosines and harm sharing

{xs}
## Saved-layer cross-check against steering (steering-equivalent definitions)

{xct}
max |Δcos(f,h)| = {XC.d_cos.abs().max():.4f}, max |ΔAUC f⊥| = {XC.d_auc_fperp.abs().max():.4f}; flagged (>0.05 cos or >0.03 AUC): {int(XC.flag.sum())}.

## Files

`b_all_layers/` (directions_all.npz, diagnostics.json/csv, cross_model.csv, manifest, DONE), `s1_labels_ja_s0/`, `qualifying_layers.json`,
`hypothesis_H_by_layer.csv`, `crosscheck_steering.csv`, `figures/*.png`.
"""
(B / "report.md").write_text(rep)
res = {"L_star": LSTAR, "negative_control": qual["negative_control"], "qualifying_layers": qual["qualifying_layers"], "counts": J["counts"], "low_n": J["low_n"],
       "train_language": TRAIN.to_dict(orient="records"), "threshold_layers": PAR, "crosscheck": XC.to_dict(orient="records"), "hardware": HW,
       "L_star_values": best}
(B / "results.json").write_text(json.dumps(res, indent=1, default=float))
files = ["report.md", "results.json", "qualifying_layers.json", "hypothesis_H_by_layer.csv", "crosscheck_steering.csv"] + \
        [f"figures/{p.name}" for p in sorted(FIG.glob("*.png"))] + [f"b_all_layers/{f}" for f in ("manifest.json", "DONE")] + \
        [f"s1_labels_ja_s0/{f}" for f in ("manifest.json", "DONE")]
man = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "experiment": "B", "agent": "agent2", **HW,
       "L_star": LSTAR, "negative_control": qual["negative_control"],
       "activations": {"local_dir": str(B / "acts") if (B / "acts").exists() else None, "vm_dir": MAN["activations"]["dir"],
                       "n_files": MAN["activations"]["n_files"], "total_bytes": MAN["activations"]["total_bytes"]},
       "files": {f: {"bytes": (B / f).stat().st_size, "sha256": sha256_file(B / f)} for f in files}}
(B / "manifest.json").write_text(json.dumps(man, indent=1))
(B / "DONE").write_text(sha256_file(B / "manifest.json") + "\n")
print(rep[:3000])
